In [ ]:
import tensorflow as tf

print("CUDA build info:")
print(tf.sysconfig.get_build_info())

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

print("Number of GPUs:", len(gpus))

for i, gpu in enumerate(gpus):
    print(f"GPU {i}:", gpu)

CUDA build info:
OrderedDict([('cpu_compiler', 'clang'), ('cuda_compute_capabilities', ['sm_60', 'sm_70', 'sm_80', 'sm_89', 'compute_90']), ('cuda_version', '12.5.1'), ('cudnn_version', '9'), ('is_cuda_build', True), ('is_rocm_build', False), ('is_tensorrt_build', False)])
Number of GPUs: 4
GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
GPU 1: PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
GPU 2: PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')
GPU 3: PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')


In [ ]:
#PATH SETUP

import os
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

BASE_PATH = "/nvme/aarthi/Work2-ABIDE/autism_macaf_project-main2"

# change working directory
os.chdir(BASE_PATH)

print("Working Directory:", os.getcwd())

# create folders if they don't exist
os.makedirs("data", exist_ok=True)
os.makedirs("data/models", exist_ok=True)

# verify files in folder
print("\nProject Files:")
print(os.listdir())

Working Directory: /nvme/aarthi/Work2-ABIDE/autism_macaf_project-main2

Project Files:
['dataset_loader.py', 'download_dataset.py', '.ipynb_checkpoints', 'requirements.txt', 'generate_npz.py', 'FC_Computation.py', 'train_test.py', 'utils.py', 'data', 'create_split.py', 'macaf_net.py', 'Phenotypic_V1_0b_preprocessed1.csv', 'README.md', 'prepare_dataset.py', 'Execution_notebook.ipynb', 'pheno_info.py', '__pycache__']


In [ ]:
#Installation of required packages
!pip install -r requirements.txt
!pip install torch h5py scikit-learn matplotlib numpy
!pip install openpyxl

In [ ]:
#Dataset download from PCP (ABIDE1)
!python download_dataset.py

In [ ]:
#Functional Connectivity Computation

!python prepare_dataset.py --folds=10 --whole cc200 aal ez ho tt dosenbach160

I0000 00:00:1775887709.820346  916066 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775887709.896868  916066 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775887711.393878  916066 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

Loading derivative: cc200
Starting pool of 1 jobs
Processing 1

In [ ]:
from dataset_loader import AutismDataset
from macaf_net import MACAFNetHybrid
from utils import site_balanced_split
from metrics import compute_metrics
from openpyxl import Workbook

In [ ]:
from dataset_loader import AutismDataset

dataset = AutismDataset("data/abide.hdf5")

print("Total subjects:", len(dataset))

sample = dataset[0]

for i,x in enumerate(sample[:-1]):
    print(i, x.shape)

print("Label:", sample[-1])

Total subjects: 1035
0 torch.Size([6670])
1 torch.Size([19900])
2 torch.Size([12880])
3 torch.Size([6105])
4 torch.Size([6670])
5 torch.Size([4656])
Label: tensor(0)


In [ ]:
import torch
from dataset_loader import AutismDataset
from macaf_net import MACAFNetHybrid

# -----------------------------
# 1️⃣ Load dataset
# -----------------------------
dataset = AutismDataset("data/abide.hdf5")
aal, cc200, dosenbach, ho, ez, tt, label = dataset[0]

# -----------------------------
# 2️⃣ Prepare input dimensions
# -----------------------------
input_dims = [x.shape[0] for x in (aal, cc200, dosenbach, ho, ez, tt)]

# -----------------------------
# 3️⃣ Initialize MACAFNetHybrid
# -----------------------------
model = MACAFNetHybrid(
    atlas_dims=input_dims,
    embed_dim=64,
    dropout=0.4,
    n_heads=4,
    transformer_layers=1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# -----------------------------
# 4️⃣ Forward pass (batch dimension)
# -----------------------------
with torch.no_grad():
    inputs = [x.unsqueeze(0).to(device) for x in (aal, cc200, dosenbach, ho, ez, tt)]
    output = model(*inputs)
    prob = torch.sigmoid(output)

# -----------------------------
# 5️⃣ Print output
# -----------------------------
print("Output shape:", output.shape)       # [1,1]
print("Logit value:", output.item())
print("Predicted probability:", prob.item())
print("True label:", label.item())

Output shape: torch.Size([1, 1])
Logit value: 0.19216981530189514
Predicted probability: 0.5478951334953308
True label: 0


In [ ]:
!python create_split.py

Total samples: 1035
Train: 724
Val: 155
Test: 156
Saved splits/data_split.npz


In [ ]:
import numpy as np

data = np.load("splits/data_split.npz")

print(data.files)

['train_idx', 'val_idx', 'test_idx']


In [ ]:
#Train
!python train_test.py

Total Experiments: 63
ALL EXPERIMENTS COMPLETED
Excel saved at: results_macafnet/all_experiments_detailed.xlsx
